# 实时调度验证（Jupyter 版，一个场景一个 cell）

每个场景 cell 会：

1. 打印当前 `f_luma/quads/u`；
2. **等你按回车**（想准备多久都行）；
3. 倒计时后跑 12 秒实时调度；
4. 打印切换次数、空间覆盖、空间保持、最终 mode/PARAM；
5. 写 `data/live_sceneN.csv`。

跑之前确保没有别的 notebook 占用摄像头/VDMA。
如果 setup 报 `Root permissions required`，用 sudo 启动 Jupyter，
或改用终端里的 `stage08_auto_schedule/auto_schedule.py`。


In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, 'lib', 'video_setup.py')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import live_scheduler_notebook as live
print('ROOT =', ROOT)
print('euid =', os.geteuid())


In [ ]:
# 摄像头已经由别的 notebook 配好时，把 skip_camera 改成 True
ol, regs, vdma0, vdma1 = live.setup(skip_camera=False, display=True)


## Preview

摆场景时反复跑这一格。


In [ ]:
live.preview(regs)


## Scene 0：均匀暗场

期望最终 `MODE=0`，PARAM 接近 255（增益拉满），不要出现 mode 1。


In [ ]:
res0 = live.run_scene(regs, 0, duration=12, wait=8)


## Scene 1：暗场 + 亮窗

期望日志出现 `空间覆盖：MLP=0 -> 模式=1`，最终 `MODE=1`。


In [ ]:
res1 = live.run_scene(regs, 1, duration=12, wait=8)


## Scene 2：均匀正常光

期望最终 `MODE=2`，空间保持次数为 0（u≈0.27 < 0.30）。


In [ ]:
res2 = live.run_scene(regs, 2, duration=12, wait=8)


## Scene 3：正常亮度 + 逆光

期望最终 `MODE=1`，空间覆盖次数大于 0。


In [ ]:
res3 = live.run_scene(regs, 3, duration=12, wait=8)


## 汇总

读 `data/live_scene*.csv`，打印四场景汇总表。


In [ ]:
live.summary_all()


## 收尾

停 VDMA、释放 overlay。


In [ ]:
live.cleanup(ol, vdma0, vdma1)
